# Question 4: Stemming and lemmatization

On the unique final tokens, compare: Porter Stemmer, Lancaster Stemmer, Regexp Stemmer, WordNet Lemmatizer with POS information from `nltk.pos_tag()`. For the Regexp Stemmer, choose a suffix based on at least two words in your own dataset and justify your choice. Create a table containing only the words for which the outputs differ.

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.tokenize import word_tokenize, sent_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
import string
import re
import pandas as pd

roll_number = "1024170030"
name = "Nikunj"
fingerprint = "E5316233"

tickets = {
    "T01": "The file is 95.5% uploaded.",
    "T02": "The LMS says 'Not Submitted'.",
    "T03": "The VPN crashes while opening. Please help.",
    "T04": "The laptop is showing an error. Please resolve it.",
    "T05": "Password is running slowly after resetting my password. I don't know what changed.",
    "T06": "The Wi-Fi works, but the VPN doesn't.",
    "T07": "Laptop stopped responding after restarting the system. Can you check this?",
    "T08": "Email is not working on my laptop. Please resolve it.",
    "T09": "Printer crashes while opening while uploading the file. Can you check this?",
    "T10": "Python is running slowly before the assignment deadline. I don't know what changed.",
    "T11": "See https://lms.thapar.edu/help for details.",
    "T12": "I cannot access the laptop on my laptop. Please resolve it."
}

stop_words = set(stopwords.words('english'))

def process_ticket(text):
    sentences = sent_tokenize(text)
    tokens = word_tokenize(text)
    punct_tokens = [t for t in tokens if t in string.punctuation]
    stop_tokens = [t for t in tokens if t.lower() in stop_words]
    final_tokens = [t.lower() for t in tokens if t not in string.punctuation and t.lower() not in stop_words]
    return {
        "sentences": sentences,
        "tokens": tokens,
        "punct_tokens": punct_tokens,
        "stop_tokens": stop_tokens,
        "final_tokens": final_tokens
    }


In [ ]:
all_final_tokens = []
for text in tickets.values():
    result = process_ticket(text)
    all_final_tokens.extend(result["final_tokens"])

unique_final_tokens = sorted(set(all_final_tokens))
unique_final_tokens

## Regexp Stemmer suffix choice

The dataset contains `uploaded` and `resetting`, both of which end in `ing` or `ed`-style suffixes once inflected forms such as `working`, `showing`, `running` and `restarting` are included. The suffix `ing$` is chosen for the Regexp Stemmer because it appears in multiple dataset words (`uploading`, `working`, `showing`, `running`, `restarting`, `resetting`) and stripping it reduces each to a recognizable root.

In [ ]:
porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp_stemmer = RegexpStemmer('ing$', min=4)
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

pos_tags = nltk.pos_tag(unique_final_tokens)

comparison_rows = []
for word, tag in pos_tags:
    porter_result = porter.stem(word)
    lancaster_result = lancaster.stem(word)
    regexp_result = regexp_stemmer.stem(word)
    lemma_result = lemmatizer.lemmatize(word, pos=get_wordnet_pos(tag))
    comparison_rows.append({
        "Word": word,
        "POS": tag,
        "Porter": porter_result,
        "Lancaster": lancaster_result,
        "Regexp": regexp_result,
        "Lemma": lemma_result
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## Only the words where outputs differ

In [ ]:
def all_same(row):
    return row["Porter"] == row["Lancaster"] == row["Regexp"] == row["Lemma"]

differing_df = comparison_df[~comparison_df.apply(all_same, axis=1)].reset_index(drop=True)
differing_df